# Preprocess the synthetic process data

The generator stores signals in interpretable process units. This notebook creates a robust-scaled modeling view while preserving run IDs, time, anomaly labels, and run-level splits. The scaler is fitted only on healthy training runs to avoid using fault or evaluation data to set the baseline. Run `0.1-validate_generated_data.ipynb` first.

In [1]:
from pathlib import Path
import sys


def find_project_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the project root from the notebook working directory.")


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

Project root: <repo>


In [2]:
import json

import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler

from carl.causalai_process import VARIABLE_NAMES


SOURCE_DIR = PROJECT_ROOT / "data" / "processed" / "causalAIProcess"
OUTPUT_DIR = PROJECT_ROOT / "data" / "interim" / "causalAIProcess"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

traces_path = SOURCE_DIR / "traces.parquet"
runs_path = SOURCE_DIR / "runs.csv"
if not traces_path.is_file() or not runs_path.is_file():
    raise FileNotFoundError(
        f"Generated traces and run manifest are required in {SOURCE_DIR}; "
        "run 0.0-generate_process_dataset.ipynb first."
    )

traces = pd.read_parquet(traces_path)
runs = pd.read_csv(runs_path)
feature_columns = list(VARIABLE_NAMES)
metadata_columns = ["run_id", "time_step", "time_seconds", "anomaly", "split"]
missing_columns = set(feature_columns + metadata_columns) - set(traces.columns)
if missing_columns:
    raise KeyError(f"Generated trace data are missing columns: {sorted(missing_columns)}")

healthy_training_run_ids = runs.loc[
    runs["split"].eq("train") & runs["fault_type"].eq("healthy"), "run_id"
].unique()
healthy_training = traces["run_id"].isin(healthy_training_run_ids)
if not healthy_training.any():
    raise ValueError("No healthy training samples are available to fit the scaler.")
if traces.loc[healthy_training, "run_id"].nunique() == 0:
    raise ValueError("No healthy training runs are available to fit the scaler.")

scaler = RobustScaler()
scaler.fit(traces.loc[healthy_training, feature_columns])
scaled_traces = traces[metadata_columns].copy()
scaled_traces[feature_columns] = scaler.transform(traces[feature_columns])
if not np.isfinite(scaled_traces[feature_columns].to_numpy()).all():
    raise ValueError("Scaling produced non-finite feature values.")

scaled_path = OUTPUT_DIR / "traces_scaled.parquet"
scaled_traces.to_parquet(scaled_path, index=False)
scaler_parameters = pd.DataFrame(
    {
        "variable": feature_columns,
        "center": scaler.center_,
        "scale": scaler.scale_,
    }
)
scaler_parameters.to_csv(OUTPUT_DIR / "scaler_parameters.csv", index=False)

preprocessing_metadata = {
    "source_directory": str(SOURCE_DIR),
    "scaler": "sklearn.preprocessing.RobustScaler",
    "fit_population": "healthy samples from training runs only",
    "fit_run_ids": sorted(healthy_training_run_ids.tolist()),
    "feature_columns": feature_columns,
    "preserved_columns": metadata_columns,
    "sample_count": len(scaled_traces),
}
(OUTPUT_DIR / "preprocessing_metadata.json").write_text(
    json.dumps(preprocessing_metadata, indent=2) + "\n",
    encoding="utf-8",
)

print(f"Wrote {scaled_path}: {scaled_traces.shape}")
print(f"Scaler fitted on {healthy_training.sum():,} healthy training samples from "
      f"{traces.loc[healthy_training, 'run_id'].nunique()} runs.")
display(scaler_parameters)

Wrote <repo>\data\interim\causalAIProcess\traces_scaled.parquet: (2000000, 11)
Scaler fitted on 350,000 healthy training samples from 350 runs.


,variable,center,scale
0,feed_flow,100.014513,8.297104
1,feed_concentration,0.999769,0.076876
2,coolant_flow,100.023912,8.310261
3,reactor_concentration,0.699801,0.027955
4,reactor_temperature,349.987783,6.323438
5,product_quality,0.849993,0.003078
